In [2]:
import numpy as np
import dgl
from dgl.nn import GraphConv
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import dgl.data

In [3]:
dataset = dgl.data.CoraGraphDataset()

  NumNodes: 2708
  NumEdges: 10556
  NumFeats: 1433
  NumClasses: 7
  NumTrainingSamples: 140
  NumValidationSamples: 500
  NumTestSamples: 1000
Done loading data from cached files.


In [6]:
dataset

Dataset("cora_v2", num_graphs=1, save_path=/home/x1113496/.dgl/cora_v2_d697a464)

In [10]:
#f = dataset[1] error

In [4]:
g = dataset[0]

In [5]:
g

Graph(num_nodes=2708, num_edges=10556,
      ndata_schemes={'feat': Scheme(shape=(1433,), dtype=torch.float32), 'label': Scheme(shape=(), dtype=torch.int64), 'test_mask': Scheme(shape=(), dtype=torch.bool), 'val_mask': Scheme(shape=(), dtype=torch.bool), 'train_mask': Scheme(shape=(), dtype=torch.bool)}
      edata_schemes={})

In [ ]:
#cora dataset
#node: 머신러닝 논문 (7개의 class)
#edge: 논문 쌍간의 인용


In [7]:
print('Number of nodes:', g.num_nodes())
print('Number of edges:', g.num_edges())

Number of nodes: 2708
Number of edges: 10556


In [11]:
print('Node feature names:', g.ndata.keys())
print('Edge feature names:', g.edata.keys())


Node feature names: dict_keys(['feat', 'label', 'test_mask', 'val_mask', 'train_mask'])
Edge feature names: dict_keys([])


In [12]:
print('Number of training nodes:', g.ndata['train_mask'].int().sum().item())
print('Number of validating nodes:', g.ndata['val_mask'].int().sum().item())
print('Number of testing nodes:', g.ndata['test_mask'].int().sum().item())

Number of training nodes: 140
Number of validating nodes: 500
Number of testing nodes: 1000


In [13]:
print('Number of classes:', (g.ndata['label'].max()+1).item())
print('Node feature shape:', g.ndata['feat'].shape)

Number of classes: 7
Node feature shape: torch.Size([2708, 1433])


In [14]:
print('Number of training nodes:', g.ndata['train_mask'].int().sum().item())
print('Number of validating nodes:', g.ndata['val_mask'].int().sum().item())
print('Number of testing nodes:', g.ndata['test_mask'].int().sum().item())

print('Number of classes:', (g.ndata['label'].max()+1).item())
print('Node feature shape:', g.ndata['feat'].shape)

Number of training nodes: 140
Number of validating nodes: 500
Number of testing nodes: 1000
Number of classes: 7
Node feature shape: torch.Size([2708, 1433])


In [15]:
class GCN(nn.Module):
    def __init__(self, in_feats, n_hidden, n_classes):
        super(GCN, self).__init__()
        self.conv1 = GraphConv(in_feats, n_hidden)
        self.conv2 = GraphConv(n_hidden, n_classes)
        self.relu = nn.ReLU()
    
    def forward(self, g, in_feat):
        output = self.conv1(g, in_feat)
        output = self.relu(output)
        output = self.conv2(g, output)
        return output 

models = GCN(g.ndata['feat'].shape[1], 16, dataset.num_classes)

In [21]:
def criterion(pred_y, true_y, mask):
    pred_y = pred_y[mask]
    true_y = true_y[mask]
    return F.cross_entropy(pred_y, true_y)

def accuracy(pred_y, true_y, mask):
    pred_y = pred_y[mask]
    true_y = true_y[mask]
    return (pred_y == true_y).float().mean()

def trainer(g, model, n_epoch, device):
    optimizer = optim.Adam(model.parameters())
    
    best_val_acc = 0
    best_test_acc = 0
    g = g.to(device)
    features = g.ndata['feat'].to(device)
    labels = g.ndata['label'].to(device)
    train_mask = g.ndata['train_mask']
    val_mask = g.ndata['val_mask']
    test_mask = g.ndata['test_mask']
    
    for epoch in range(1, n_epoch+1):
        pred_y = model(g, features).to(device)
        loss = criterion(pred_y, labels, train_mask)
        pred_y = pred_y.argmax(1)
        train_acc = accuracy(pred_y, labels, val_mask)
        val_acc = accuracy(pred_y, labels, val_mask)
        test_acc = accuracy(pred_y, labels, test_mask)
        
        if best_val_acc < val_acc:
            best_val_acc = val_acc
            vest_test_acc = test_acc
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if epoch %  10 == 0 :
            print(f'In epoch [{epoch}/{n_epoch}]\t loss: {loss:.3f}\t train_acc: {train_acc*100:.2f}%\t val_acc: {val_acc*100:.2f}%\t test_acc: {test_acc*100:.2f}%')
            print(f'best val acc: {best_val_acc*100:.2f}%\t best test acc: {best_test_acc*100:.2f}%\n')

In [22]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = GCN(g.ndata['feat'].shape[1], 16, dataset.num_classes).to(device)
trainer(g, model, 100, device)

In epoch [10/100]	 loss: 1.937	 train_acc: 43.00%	 val_acc: 43.00%	 test_acc: 37.30%
best val acc: 43.80%	 best test acc: 0.00%

In epoch [20/100]	 loss: 1.924	 train_acc: 51.60%	 val_acc: 51.60%	 test_acc: 51.60%
best val acc: 51.60%	 best test acc: 0.00%

In epoch [30/100]	 loss: 1.908	 train_acc: 64.80%	 val_acc: 64.80%	 test_acc: 65.60%
best val acc: 65.20%	 best test acc: 0.00%

In epoch [40/100]	 loss: 1.890	 train_acc: 65.80%	 val_acc: 65.80%	 test_acc: 66.80%
best val acc: 67.20%	 best test acc: 0.00%

In epoch [50/100]	 loss: 1.871	 train_acc: 65.40%	 val_acc: 65.40%	 test_acc: 67.10%
best val acc: 67.20%	 best test acc: 0.00%

In epoch [60/100]	 loss: 1.850	 train_acc: 66.00%	 val_acc: 66.00%	 test_acc: 67.70%
best val acc: 67.20%	 best test acc: 0.00%

In epoch [70/100]	 loss: 1.827	 train_acc: 67.00%	 val_acc: 67.00%	 test_acc: 67.80%
best val acc: 67.20%	 best test acc: 0.00%

In epoch [80/100]	 loss: 1.803	 train_acc: 67.40%	 val_acc: 67.40%	 test_acc: 68.70%
best val acc